# Reconstruction candidate du nettoyage

Cette expérience nettoie les 1 186 documents dans un nouveau dossier horodaté. Elle ne modifie ni `data/cleaned`, ni les chunks, ni LanceDB, ni BM25.


In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime
import sys
import frontmatter
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import load_config
from cleaner import HTMLCleaner

cfg = load_config(ROOT / 'config.yaml')
candidate_dir = ROOT / 'data' / 'experiments' / f"cleaned_candidate_{datetime.now():%Y%m%d_%H%M%S}"
cleaner = HTMLCleaner(cfg)
cleaner.cleaned_dir = candidate_dir
candidate_dir.mkdir(parents=True, exist_ok=True)
print(f'Dossier candidat : {candidate_dir}')


Dossier candidat : <PROJECT_ROOT>\data\experiments\cleaned_candidate_20260803_121334


In [2]:
candidate_docs = cleaner.clean_all()
method_counts = Counter(doc.extraction_method for doc in candidate_docs)
display(pd.Series(method_counts, name='documents').sort_values(ascending=False).to_frame())
print(f'Documents candidats : {len(candidate_docs)}')


,documents
trafilatura_precision,689
structured_main_richer,162
pdf_text,74
structured_main_fallback,7


Documents candidats : 932


In [3]:
current_dir = cfg.project.data_path / 'cleaned'
comparison = []
for doc in candidate_docs:
    current_path = current_dir / f'{doc.doc_id}.md'
    current_body = frontmatter.load(current_path).content if current_path.exists() else ''
    old_chars = len(' '.join(current_body.split()))
    new_chars = len(' '.join(doc.markdown.split()))
    comparison.append({
        'doc_id': doc.doc_id, 'title': doc.title, 'url': doc.url,
        'method': doc.extraction_method, 'old_chars': old_chars,
        'new_chars': new_chars, 'delta_chars': new_chars - old_chars,
        'ratio': new_chars / max(old_chars, 1),
        'body_changed': current_body.strip() != doc.markdown.strip(),
    })
comparison_df = pd.DataFrame(comparison)
summary = pd.Series({
    'documents': len(comparison_df),
    'corps_modifiés': int(comparison_df['body_changed'].sum()),
    'fallbacks_enrichis': int(comparison_df['method'].eq('structured_main_richer').sum()),
    'nouveaux_documents': int(comparison_df['old_chars'].eq(0).sum()),
    'caractères_avant': int(comparison_df['old_chars'].sum()),
    'caractères_après': int(comparison_df['new_chars'].sum()),
})
display(summary.to_frame('valeur'))


,valeur
documents,932
corps_modifiés,263
fallbacks_enrichis,162
nouveaux_documents,13
caractères_avant,4327342
caractères_après,4796883


In [4]:
fallbacks = comparison_df[comparison_df['method'].eq('structured_main_richer')]
display(fallbacks.sort_values('delta_chars', ascending=False)[
    ['title', 'url', 'old_chars', 'new_chars', 'delta_chars', 'ratio']
].head(30))


,title,url,old_chars,new_chars,delta_chars,ratio
811,FAQ - ENSEA,https://ensea.ed.ci/faq/,704,17438,16734,24.769886
411,INNOVATION ET ENTREPRENEURIAT - ENSEA,https://ensea.ed.ci/partenaires/innovation-et-...,3683,17075,13392,4.636166
819,Master en Statistique Agricole - ENSEA,https://ensea.ed.ci/formation-initiale/master-...,845,11364,10519,13.448521
393,Master in Agricultural Statistics - ENSEA,https://ensea.ed.ci/initial-education/master-i...,881,10849,9968,12.314415
707,Master in Agricultural Statistics - ENSEA,https://ensea.ed.ci/initial-education/master-i...,881,10849,9968,12.314415
486,Notre équipe - ENSEA,https://ensea.ed.ci/notre-equipe/,537,9572,9035,17.824953
756,Chaire de l’Actuariat - ENSEA,https://ensea.ed.ci/formation-initiale/chaire-...,2204,11221,9017,5.091198
3,Filière des Ingénieurs Statisticiens Economist...,https://ensea.ed.ci/formation-initiale/ise/,685,9620,8935,14.043796
283,Chaire de l’Actuariat - ENSEA,https://ensea.ed.ci/formation-initiale/chaire-...,1739,10484,8745,6.028752
392,LISTE DES HOTELS AFSS 2025 - ENSEA,https://ensea.ed.ci/liste-des-hotels-pour-afes/,1037,9599,8562,9.256509


In [5]:
doctorants = next(doc for doc in candidate_docs if doc.url == 'https://ensea.ed.ci/formation-doctorale/doctorants/')
partners = next(doc for doc in candidate_docs if doc.url == 'https://ensea.ed.ci/cooperation/partenaires-de-recherche/')
assert doctorants.extraction_method == 'structured_main_richer'
assert 'Alain NDIKUMANA' in doctorants.markdown
assert partners.extraction_method == 'structured_main_richer'
assert 'INSD-Burkina Faso' in partners.markdown
assert not any(doc.url.lower().split('?')[0].endswith(('.jpg', '.jpeg', '.png', '.gif', '.webp')) for doc in candidate_docs)
assert not fallbacks['url'].str.contains(r'/page/\d+/?$', regex=True).any()
assert (cfg.project.data_path / 'chunks' / 'chunks.jsonl').exists()
print('Candidate validé. Le corpus, les chunks et les index réels sont restés intacts.')
print(f'Résultat isolé disponible dans : {candidate_dir}')


Candidate validé. Le corpus, les chunks et les index réels sont restés intacts.
Résultat isolé disponible dans : <PROJECT_ROOT>\data\experiments\cleaned_candidate_20260803_121334
